In [14]:
import zipfile
import os

In [3]:
# Unzip the dataset
with zipfile.ZipFile('keratoconus.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset')

In [ ]:
# This identifies the correct folder path for training
data_path = 'dataset/Independent Test Set/Independent Test Set'
print("Folders found:", os.listdir(data_path))

Folders found: ['Suspect', 'Keratoconus', 'Normal']


In [12]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

In [6]:
# 1. IMAGE PREPARATION
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = datagen.flow_from_directory(
    data_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    data_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 840 images belonging to 3 classes.
Found 210 images belonging to 3 classes.


In [ ]:
# 2. THE MODEL (The AI Brain)

base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2), # Prevents the AI from 'memorizing' instead of 'learning'
    layers.Dense(3, activation='softmax') # 3 outputs: Normal, Suspect, Keratoconus
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# 3. TRAINING
print("Starting training...")
model.fit(train_gen, validation_data=val_gen, epochs=10)

In [13]:
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

In [ ]:
def predict_eye_issue(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)
    classes = list(train_gen.class_indices.keys()) # ['Keratoconus', 'Normal', 'Suspect']
    
    result = classes[np.argmax(prediction)]
    confidence = np.max(prediction) * 100
    
    print(f"Prediction: {result} ({confidence:.2f}% confidence)")
    plt.imshow(img)
    plt.axis('off')
    plt.show()

# Replace this with the filename of your uploaded eye report
# predict_eye_issue('img path.jpeg')